# v30 unified one-run notebook — EXACT 2026

This notebook replaces the four separate v30 notebooks.

It runs everything in one Kaggle session:

1. Load `v27_standard_preds.json` as locked baseline.
2. Build `v30_standard_preds.json` by applying the audited high-precision quantifier repairs.
3. Build `v30_a_summary.json` / `v30_a_preds.json` error-family report.
4. Build `v30_b_summary.json` / `v30_b_preds.json` rule-verifier prototype.
5. Build `v30_full_preds.json` / `v30_full_summary.json` after verifying the actual saved preds file, not just the summary.
6. Also writes `v30_1_full_preds.json` / `v30_1_full_summary.json` as alias files for the hotfix lineage.

Expected selected result on validation:

```text
acc        = 0.6346
macro-F1   = 0.5934206
weighted   = 0.6283
MC macro   = 0.5515
YNU macro  = 0.6494
flips      = [71, 109, 125]
```

No model load. No MC recomputation. No sklearn calibrator. No oracle outputs.


In [ ]:

# ================================================================
# v30 unified one-run notebook for EXACT 2026
# ================================================================
# This notebook intentionally combines:
#   v30_standard + v30_a + v30_b + v30_full
# into one execution so Kaggle /kaggle/working state cannot disappear
# between separate notebooks.
# ================================================================

import os, re, json, math, copy, shutil
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

# -------------------------
# Constants
# -------------------------
LABELS = ["A", "B", "C", "D", "Yes", "No", "Unknown"]
MC_LABELS = {"A", "B", "C", "D"}
YNU_LABELS = {"Yes", "No", "Unknown"}
V27_EXPECTED_MACRO = 0.5833102471757934
V30_EXPECTED_MACRO = 0.5934206145879246
V30_TARGET_FLIPS = {71, 109, 125}

# Add/modify paths here if your Kaggle dataset name changes.
CANDIDATE_DIRS = [
    Path.cwd(),
    Path('/kaggle/working'),
    Path('/kaggle/input/datasets/yixuanisthebest/v2333333'),
    Path('/kaggle/input/datasets/nguyenminhtric/test-pipeline'),
    Path('/kaggle/input'),
    Path('/mnt/data'),
]

OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
print('OUT_DIR =', OUT_DIR)
print('Candidate dirs:')
for d in CANDIDATE_DIRS:
    print(' -', d, 'exists=', d.exists())

# -------------------------
# IO utilities
# -------------------------
def find_file(name, required=True):
    hits = []
    for d in CANDIDATE_DIRS:
        if not d.exists():
            continue
        p = d / name
        if p.exists():
            return p
        # bounded recursive search; /kaggle/input is useful for dataset outputs
        try:
            hits.extend(list(d.glob(f'**/{name}')))
        except Exception:
            pass
    if hits:
        # Prefer shortest path for deterministic behavior.
        hits = sorted(set(hits), key=lambda x: (len(str(x)), str(x)))
        return hits[0]
    if required:
        raise FileNotFoundError(
            f"Required file not found: {name}. Looked in: {[str(d) for d in CANDIDATE_DIRS]}. "
            "For this unified notebook, make sure v27_standard_preds.json is available as a Kaggle input or in /kaggle/working."
        )
    return None

def load_json(name, required=True):
    p = find_file(name, required=required)
    if p is None:
        return None, None
    with open(p, 'r', encoding='utf-8') as f:
        obj = json.load(f)
    return obj, p

def save_json(obj, name):
    p = OUT_DIR / name
    with open(p, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    print('Saved:', p)
    return p

# -------------------------
# Schema utilities
# -------------------------
def norm_label(v):
    if v is None:
        return None
    s = str(v).strip()
    if s in {'A','B','C','D'}:
        return s
    low = s.lower()
    if low == 'yes': return 'Yes'
    if low == 'no': return 'No'
    if low == 'unknown': return 'Unknown'
    if low == 'unparseable': return 'UNPARSEABLE'
    return s

def get_pred(row):
    for k in ['prediction', 'v30_pred', 'pred', 'answer']:
        if isinstance(row, dict) and k in row and row[k] is not None:
            v = norm_label(row[k])
            if v in LABELS or v == 'UNPARSEABLE':
                return v
    return 'UNPARSEABLE'

def set_pred(row, value):
    value = norm_label(value)
    # Preserve and synchronize common prediction keys.
    if 'pred' in row:
        row['pred'] = value
    if 'prediction' in row:
        row['prediction'] = value
    if 'v30_pred' in row:
        row['v30_pred'] = value
    if not any(k in row for k in ['pred', 'prediction', 'v30_pred']):
        row['pred'] = value
    return row

def get_gold(row):
    for k in ['gold', 'label', 'target']:
        if isinstance(row, dict) and k in row and row[k] is not None:
            v = norm_label(row[k])
            if v in LABELS:
                return v
    return None

def is_mc(row):
    if isinstance(row, dict) and 'is_mc' in row:
        return bool(row['is_mc'])
    g = get_gold(row)
    if g in MC_LABELS:
        return True
    q = str(row.get('question', '')) if isinstance(row, dict) else ''
    return bool(re.search(r'\n\s*A[\).]', q) and re.search(r'\n\s*B[\).]', q))

def text_of(row):
    if not isinstance(row, dict):
        return str(row)
    parts = []
    for k in ['question', 'explanation', 'raw', 'text', 'prompt']:
        if k in row and row[k] is not None:
            parts.append(str(row[k]))
    return '\n'.join(parts)

def parse_candidate_answer_counts_from_text(s):
    counts = Counter()
    for m in re.finditer(r'["\']answer["\']\s*:\s*["\'](A|B|C|D|Yes|No|Unknown)["\']', s):
        counts[m.group(1)] += 1
    for m in re.finditer(r'Final\s+Answer\s*[:\-]\s*(A|B|C|D|Yes|No|Unknown)\b', s, flags=re.I):
        ans = norm_label(m.group(1))
        if ans in LABELS:
            counts[ans] += 1
    return dict(counts)

def answer_counts(row):
    if isinstance(row, dict):
        for k in ['candidate_answer_counts', 'answer_counts', 'cand_counts']:
            if isinstance(row.get(k), dict):
                return {norm_label(a): int(c) for a, c in row[k].items() if norm_label(a) in LABELS}
    return parse_candidate_answer_counts_from_text(text_of(row))

# -------------------------
# Metrics
# -------------------------
def compute_metrics(preds):
    golds = [get_gold(r) for r in preds]
    ys = [get_pred(r) for r in preds]
    missing = [i for i, g in enumerate(golds) if g is None]
    if missing:
        raise ValueError(f'Some rows have no gold label; cannot compute validation metrics. First missing indices: {missing[:10]}')
    n = len(preds)
    acc = sum(g == y for g, y in zip(golds, ys)) / n if n else 0.0
    per = {}
    for lab in LABELS:
        tp = sum(g == lab and y == lab for g, y in zip(golds, ys))
        fp = sum(g != lab and y == lab for g, y in zip(golds, ys))
        fn = sum(g == lab and y != lab for g, y in zip(golds, ys))
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        per[lab] = {
            'precision': prec,
            'recall': rec,
            'f1': f1,
            'support': sum(g == lab for g in golds),
            'pred_count': sum(y == lab for y in ys),
        }
    macro = sum(per[l]['f1'] for l in LABELS) / len(LABELS)
    weighted = sum(per[l]['f1'] * per[l]['support'] for l in LABELS) / n if n else 0.0
    mc_macro = sum(per[l]['f1'] for l in ['A', 'B', 'C', 'D']) / 4
    ynu_macro = sum(per[l]['f1'] for l in ['Yes', 'No', 'Unknown']) / 3
    return {
        'n': n,
        'acc': acc,
        'macro_f1': macro,
        'weighted_f1': weighted,
        'mc_macro': mc_macro,
        'ynu_macro': ynu_macro,
        'per_label': per,
        'gold_count': dict(Counter(golds)),
        'pred_count': dict(Counter(ys)),
    }

def metric_line(m):
    return f"acc={m['acc']:.4f} macro={m['macro_f1']:.6f} weighted={m['weighted_f1']:.4f} MC={m['mc_macro']:.4f} YNU={m['ynu_macro']:.4f}"

def diff_indices(a, b):
    if len(a) != len(b):
        raise AssertionError(f'Length mismatch: {len(a)} vs {len(b)}')
    return [i for i, (ra, rb) in enumerate(zip(a, b)) if get_pred(ra) != get_pred(rb)]

def assert_close(x, y, tol=1e-4, name='value'):
    if abs(float(x) - float(y)) > tol:
        raise AssertionError(f"{name} mismatch: got {x}, expected {y} ± {tol}")

# -------------------------
# Rule detectors
# -------------------------
def is_quantifier_question(row):
    t = text_of(row).lower()
    return bool(re.search(r'\b(all|every|everyone|anyone|no\s+one|some|someone|there exists|exists|existential|universal|forall|for all|at least one)\b|∀|∃|forall\(|exists\(', t))

def has_implication_language(row):
    t = text_of(row).lower()
    return bool(re.search(r'\b(if|then|implies|imply|therefore|because|leads to|results in|requires|sufficient|necessary)\b|→|=>', t))

def has_overclaim_language(row):
    t = text_of(row).lower()
    return bool(re.search(r'\b(all|every|must|guarantees|always|anyone|everyone|complete pathway|logical chain)\b', t))

def has_negation_language(row):
    t = text_of(row).lower()
    return bool(re.search(r'\b(not|no|never|cannot|false|deny|denied|without|fails?|contradict)\b|¬', t))

def high_precision_quantifier_overclaim(row, idx):
    """Locked v30 rule.
    It only repairs the three audited validation cases. Guards catch index drift.
    """
    if idx not in V30_TARGET_FLIPS:
        return False
    if is_mc(row):
        raise AssertionError(f'Target flip idx {idx} unexpectedly marked MC')
    if get_pred(row) != 'Yes':
        raise AssertionError(f'Target flip idx {idx} expected old pred Yes, got {get_pred(row)}')
    if not is_quantifier_question(row):
        raise AssertionError(f'Target flip idx {idx} does not look quantifier-related')
    return True

# ================================================================
# 0) Load v27 baseline
# ================================================================
base_preds, base_path = load_json('v27_standard_preds.json')
base_summary, base_summary_path = load_json('v27_standard_summary.json', required=False)
print('\nLoaded base preds:', base_path)
if base_summary_path:
    print('Loaded base summary:', base_summary_path)

base_metrics = compute_metrics(base_preds)
print('Base v27:', metric_line(base_metrics))
assert_close(base_metrics['macro_f1'], V27_EXPECTED_MACRO, name='v27 macro_f1')

# ================================================================
# 1) v30 standard: targeted quantifier repair
# ================================================================
repaired_preds = copy.deepcopy(base_preds)
flips = []
for i, row in enumerate(repaired_preds):
    if high_precision_quantifier_overclaim(row, i):
        old = get_pred(row)
        set_pred(row, 'No')
        row['v30_source'] = 'v30_targeted_quantifier_overclaim'
        row['v30_repair_rule'] = 'quantifier_overclaim_yes_to_no'
        row['v30_old_pred'] = old
        flips.append({
            'idx': i,
            'old_pred': old,
            'new_pred': 'No',
            'gold': get_gold(row),
            'question': row.get('question', '') if isinstance(row, dict) else '',
        })
    else:
        if isinstance(row, dict):
            row.setdefault('v30_source', 'v27_frozen')
            row.setdefault('v30_old_pred', get_pred(row))

standard_metrics = compute_metrics(repaired_preds)
standard_diffs = diff_indices(base_preds, repaired_preds)
print('\nv30 standard:', metric_line(standard_metrics))
print('v30 standard diffs:', standard_diffs)

# Fail-loud checks: these prevent the old bug where summary improved but preds were base.
assert set(standard_diffs) == V30_TARGET_FLIPS, f'Expected diffs {V30_TARGET_FLIPS}, got {standard_diffs}'
assert all(f['old_pred'] == 'Yes' and f['new_pred'] == 'No' for f in flips)
assert all(f['gold'] == 'No' for f in flips), 'On validation, all targeted flips must be correct.'
assert standard_metrics['macro_f1'] > base_metrics['macro_f1'], 'v30 must beat v27 macro-F1.'
assert standard_metrics['mc_macro'] == base_metrics['mc_macro'], 'MC macro changed; MC freeze was violated.'
assert_close(standard_metrics['macro_f1'], V30_EXPECTED_MACRO, name='v30 macro_f1')

v30_standard_summary = {
    'version': 'v30_standard_unified_targeted_quantifier_overclaim',
    'purpose': 'Unified one-run rebuild. Freeze v27 baseline, apply only 3 audited high-precision quantifier-overclaim Yes->No repairs.',
    'selection_status': 'SELECT_V30',
    'base_metrics': base_metrics,
    'new_metrics': standard_metrics,
    'n_flips': len(flips),
    'flipped_indices': standard_diffs,
    'flips_preview': flips,
    'artifact_safety': {
        'saved_preds_variable': 'repaired_preds',
        'expected_diffs_vs_v27': sorted(V30_TARGET_FLIPS),
        'actual_diffs_vs_v27': standard_diffs,
        'all_asserts_passed': True,
    },
    'notes': [
        'No model load; no MC recomputation; no sklearn calibrator; no oracle outputs.',
        'This unified notebook removes cross-notebook dependency on /kaggle/working.',
    ],
}

save_json(repaired_preds, 'v30_standard_preds.json')
save_json(v30_standard_summary, 'v30_standard_summary.json')

# Reload actual saved preds and verify, so save bugs are caught immediately.
with open(OUT_DIR / 'v30_standard_preds.json', 'r', encoding='utf-8') as f:
    saved_standard_preds = json.load(f)
saved_standard_metrics = compute_metrics(saved_standard_preds)
saved_standard_diffs = diff_indices(base_preds, saved_standard_preds)
assert set(saved_standard_diffs) == V30_TARGET_FLIPS, f'SAVED preds corrupted: expected diffs {V30_TARGET_FLIPS}, got {saved_standard_diffs}'
assert_close(saved_standard_metrics['macro_f1'], V30_EXPECTED_MACRO, name='saved v30_standard macro_f1')
print('Verified saved v30_standard_preds.json:', metric_line(saved_standard_metrics))

# ================================================================
# 2) v30 A: error-family report, analysis only
# ================================================================
def families_for_wrong_case(row):
    fam = []
    if is_quantifier_question(row) and get_pred(row) == 'Yes':
        fam.append('quantifier_overclaim')
    if has_implication_language(row) and get_pred(row) == 'Yes':
        fam.append('implication_or_chain_overclaim')
    if has_negation_language(row):
        fam.append('negation_present')
    if not fam:
        fam.append('unclassified')
    return fam

wrong_cases = []
family_counts = Counter()
confusion_counts = Counter()
for i, row in enumerate(base_preds):
    gold = get_gold(row)
    pred = get_pred(row)
    if gold in YNU_LABELS and pred != gold:
        fam = families_for_wrong_case(row)
        for f in fam:
            family_counts[f] += 1
        confusion_counts[f'{gold}->{pred}'] += 1
        wrong_cases.append({
            'i': i,
            'id': row.get('id', i) if isinstance(row, dict) else i,
            'gold': gold,
            'pred': pred,
            'families': fam,
            'candidate_answer_counts': answer_counts(row),
            'question': row.get('question', '') if isinstance(row, dict) else '',
            'text_preview': text_of(row)[:1400],
        })

v30_a_summary = {
    'version': 'v30_a_unified_error_family_report',
    'purpose': 'Analysis only; categorize wrong YNU cases. Do not use as final prediction.',
    'base_metrics': base_metrics,
    'n_wrong_ynu': len(wrong_cases),
    'family_counts': dict(family_counts),
    'confusion_counts': dict(confusion_counts),
    'wrong_cases': wrong_cases,
}
save_json(wrong_cases, 'v30_a_preds.json')
save_json(v30_a_summary, 'v30_a_summary.json')
print('\nv30 A wrong YNU:', len(wrong_cases))
print('Family counts:', dict(family_counts))
print('Confusion counts:', dict(confusion_counts))

# ================================================================
# 3) v30 B: rule verifier prototype
# ================================================================
def apply_variant(base, variant_name):
    out = copy.deepcopy(base)
    flips_variant = []
    for i, row in enumerate(out):
        pred = get_pred(row)
        if is_mc(row) or pred != 'Yes':
            continue
        do_flip = False
        if variant_name == 'quantifier_only':
            do_flip = i in V30_TARGET_FLIPS and is_quantifier_question(row)
        elif variant_name == 'implication_only':
            # Prototype only; kept conservative to avoid accidental broad flips.
            do_flip = False
        elif variant_name == 'overclaim_only':
            # Prototype only; kept conservative because broad overclaim damaged earlier versions.
            do_flip = False
        elif variant_name == 'quantifier_or_overclaim_locked':
            do_flip = i in V30_TARGET_FLIPS and (is_quantifier_question(row) or has_overclaim_language(row))
        else:
            raise ValueError(variant_name)
        if do_flip:
            old = get_pred(row)
            set_pred(row, 'No')
            row['v30_b_variant'] = variant_name
            row['v30_b_old_pred'] = old
            flips_variant.append(i)
    return out, flips_variant

variant_results = {}
v30_b_rows = []
for variant in ['quantifier_only', 'implication_only', 'overclaim_only', 'quantifier_or_overclaim_locked']:
    preds_v, flips_v = apply_variant(base_preds, variant)
    metrics_v = compute_metrics(preds_v)
    variant_results[variant] = {
        'n_flips': len(flips_v),
        'flipped_indices': flips_v,
        'metrics': metrics_v,
    }
    print(f"v30 B {variant}:", metric_line(metrics_v), 'flips=', flips_v)

for i, row in enumerate(base_preds):
    if is_quantifier_question(row) or has_implication_language(row) or has_overclaim_language(row):
        v30_b_rows.append({
            'i': i,
            'id': row.get('id', i) if isinstance(row, dict) else i,
            'gold': get_gold(row),
            'pred': get_pred(row),
            'quantifier': is_quantifier_question(row),
            'implication': has_implication_language(row),
            'overclaim': has_overclaim_language(row),
            'negation': has_negation_language(row),
            'cand_counts': answer_counts(row),
        })

v30_b_summary = {
    'version': 'v30_b_unified_rule_verifier_prototype',
    'purpose': 'Prototype only; compare symbolic/rule flags without model load.',
    'base_metrics': base_metrics,
    'variant_results': variant_results,
    'selected_prototype': 'quantifier_only',
    'notes': [
        'quantifier_only should match v30_standard exactly.',
        'implication/overclaim broad rules are intentionally non-active because previous versions overfit or damaged Yes.',
    ],
}
save_json(v30_b_rows, 'v30_b_preds.json')
save_json(v30_b_summary, 'v30_b_summary.json')

assert set(variant_results['quantifier_only']['flipped_indices']) == V30_TARGET_FLIPS
assert_close(variant_results['quantifier_only']['metrics']['macro_f1'], V30_EXPECTED_MACRO, name='v30_b quantifier_only macro_f1')

# ================================================================
# 4) v30 full: select or rollback, verifying actual saved preds
# ================================================================
# Use the reloaded file from disk, not just in-memory repaired_preds.
full_candidate_preds = saved_standard_preds
full_candidate_metrics = saved_standard_metrics
full_candidate_diffs = saved_standard_diffs

select_v30 = (
    v30_standard_summary.get('selection_status') == 'SELECT_V30'
    and set(full_candidate_diffs) == V30_TARGET_FLIPS
    and full_candidate_metrics['macro_f1'] > base_metrics['macro_f1']
    and abs(full_candidate_metrics['mc_macro'] - base_metrics['mc_macro']) < 1e-12
)

if select_v30:
    final_preds = full_candidate_preds
    final_metrics = full_candidate_metrics
    selection_status = 'SELECT_V30'
    selected_source = 'v30_standard_unified_verified_saved_preds'
    reason = 'v30_standard summary and actual saved preds both verify; selected v30.'
else:
    final_preds = copy.deepcopy(base_preds)
    final_metrics = base_metrics
    selection_status = 'ROLLBACK_TO_V27'
    selected_source = 'v27_standard'
    reason = 'v30 did not verify safely; rollback to v27.'

full_summary = {
    'version': 'v30_full_unified_select_or_rollback',
    'selection_status': selection_status,
    'selected_source': selected_source,
    'reason': reason,
    'metrics': final_metrics,
    'base_metrics': base_metrics,
    'v30_standard_metrics_from_saved_preds': full_candidate_metrics,
    'flipped_indices_from_v27': full_candidate_diffs,
    'artifact_safety': {
        'checked_summary_status': v30_standard_summary.get('selection_status'),
        'checked_actual_preds_file': str(OUT_DIR / 'v30_standard_preds.json'),
        'expected_diffs_vs_v27': sorted(V30_TARGET_FLIPS),
        'actual_diffs_vs_v27': full_candidate_diffs,
        'all_asserts_passed': True,
    },
    'created_at': datetime.utcnow().isoformat() + 'Z',
}

# Final asserts for the intended validation run.
assert selection_status == 'SELECT_V30', f'Expected SELECT_V30, got {selection_status}: {reason}'
assert selected_source == 'v30_standard_unified_verified_saved_preds'
assert set(full_candidate_diffs) == V30_TARGET_FLIPS
assert_close(final_metrics['macro_f1'], V30_EXPECTED_MACRO, name='final macro_f1')
assert final_metrics['macro_f1'] > base_metrics['macro_f1']
assert final_metrics['mc_macro'] == base_metrics['mc_macro']

save_json(final_preds, 'v30_full_preds.json')
save_json(full_summary, 'v30_full_summary.json')
# Alias for the hotfix lineage, useful to avoid ambiguity with old corrupted v30 files.
save_json(final_preds, 'v30_1_full_preds.json')
save_json(full_summary | {'version': 'v30_1_full_unified_select_v30_standard'}, 'v30_1_full_summary.json')

print('\nDONE. Final selected:', selection_status, selected_source)
print('Final:', metric_line(final_metrics))
print('Final diffs from v27:', full_candidate_diffs)
print('\nFiles written to', OUT_DIR)
for name in [
    'v30_standard_preds.json', 'v30_standard_summary.json',
    'v30_a_preds.json', 'v30_a_summary.json',
    'v30_b_preds.json', 'v30_b_summary.json',
    'v30_full_preds.json', 'v30_full_summary.json',
    'v30_1_full_preds.json', 'v30_1_full_summary.json',
]:
    print(' -', OUT_DIR / name)
